In [5]:
import os

def find_chromedriver(start_path="C:\\"):
    for root, dirs, files in os.walk(start_path):
        if "chromedriver.exe" in files:
            return os.path.join(root, "chromedriver.exe")
    return None

path = find_chromedriver()
if path:
    print("Found chromedriver at:", path)
else:
    print("chromedriver.exe not found. Please download it from https://chromedriver.chromium.org/downloads")




Found chromedriver at: C:\Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.68\chromedriver.exe


In [15]:
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from pathlib import Path

# Setup ChromeDriver
driver_path = r"C:\Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.68\chromedriver.exe"
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(driver_path), options=options)

# Load the TenderTiger page
url = "https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=odisha-tenders"
driver.get(url)
time.sleep(5)  # Allow page to fully render

# Get full page HTML and close browser
html = driver.page_source
driver.quit()

# Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# Find all <div> blocks that include the string "TID:"
blocks = soup.find_all("div", string=re.compile("TID:"))

print(f"\n🔍 Found {len(blocks)} tender blocks with 'TID:'")

tenders = []

# Go broader: collect parent containers that contain "TID:"
for tag in soup.find_all("div"):
    text = tag.get_text(separator="\n").strip()
    if "TID:" not in text:
        continue

    # Now parse the tender block
    tid_match = re.search(r"TID[:\s]*(\d+)", text)
    location_match = re.search(r"TID:\d+\s+([^\n]+)", text)
    desc_match = re.search(r"Tender Invited For(.*?)\n", text, re.DOTALL)
    financials_match = re.search(r"Worth\s*:(.*?)EMD\s*:(.*?)Due Date\s*:(.*?)(\n|$)", text)

    if tid_match:
        tid = tid_match.group(1).strip()
        location = location_match.group(1).strip() if location_match else ""
        description = desc_match.group(1).strip() if desc_match else ""
        worth = financials_match.group(1).strip() if financials_match else ""
        emd = financials_match.group(2).strip() if financials_match else ""
        due_date = financials_match.group(3).strip() if financials_match else ""

        tenders.append({
            "TID": tid,
            "Location": location,
            "Description": description,
            "Worth": worth,
            "EMD": emd,
            "Due Date": due_date
        })
        print(f"✅ Parsed TID: {tid} | Desc: {description[:30]}...")

# Save result
if tenders:
    df = pd.DataFrame(tenders)
    df.to_excel("odisha_tenders_final.xlsx", index=False)
    print(f"\n✅ Extracted {len(tenders)} tenders. Saved to 'odisha_tenders_final.xlsx'")
else:
    print("\n⚠️ No tender entries found in parsed page.")



🔍 Found 0 tender blocks with 'TID:'
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83934997 | Desc: Manpower Outsourcing Services ...
✅ Parsed TID: 83973694 | Desc: laboratory press (q3)...
✅ Parsed TID: 83973694 | Desc: laboratory press (q3)...
✅ Parsed TID: 83927580 | Desc: Allen Key set of 12 pieces, Ca...
✅ Pars

In [16]:
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from pathlib import Path

# ChromeDriver Setup
driver_path = r"C:\Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.68\chromedriver.exe"
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(driver_path), options=options)

# Open TenderTiger URL
url = "https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=odisha-tenders"
driver.get(url)
time.sleep(5)
html = driver.page_source
driver.quit()

# Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
tags = soup.find_all("div")
tenders = []
seen_tids = set()

print(f"🔍 Scanning {len(tags)} <div> blocks...")

for tag in tags:
    text = tag.get_text(separator="\n").strip()
    if "TID:" not in text:
        continue

    # Get fields
    tid_match = re.search(r"TID[:\s]*(\d+)", text)
    location_match = re.search(r"TID:\d+\s+([^\n]+)", text)
    desc_match = re.search(r"Tender Invited For(.*?)\n", text, re.DOTALL)
    financials_match = re.search(r"Worth\s*:([^\nEMD]+)EMD\s*:([^\nDue]+)Due Date\s*:([^\n]+)", text)

    if tid_match:
        tid = tid_match.group(1).strip()

        # Deduplicate
        if tid in seen_tids:
            continue
        seen_tids.add(tid)

        location = location_match.group(1).strip() if location_match else ""
        description = desc_match.group(1).strip() if desc_match else ""
        worth = financials_match.group(1).strip() if financials_match else ""
        emd = financials_match.group(2).strip() if financials_match else ""
        due_date = financials_match.group(3).strip() if financials_match else ""

        tenders.append({
            "TID": tid,
            "Location": location,
            "Description": description,
            "Worth": worth,
            "EMD": emd,
            "Due Date": due_date
        })
        print(f"✅ Parsed TID: {tid} | EMD: {emd} | Due: {due_date}")

# Save to Excel
if tenders:
    df = pd.DataFrame(tenders)
    df.to_excel("odisha_tenders_final_clean.xlsx", index=False)
    print(f"\n✅ Extracted {len(tenders)} unique tenders → saved to 'odisha_tenders_final_clean.xlsx'")
else:
    print("⚠️ No tender data found.")


🔍 Scanning 2507 <div> blocks...
✅ Parsed TID: 83934997 | EMD:  | Due: 
✅ Parsed TID: 83973694 | EMD:  | Due: 
✅ Parsed TID: 83927580 | EMD:  | Due: 
✅ Parsed TID: 84627731 | EMD:  | Due: 
✅ Parsed TID: 84037764 | EMD:  | Due: 
✅ Parsed TID: 84359961 | EMD:  | Due: 
✅ Parsed TID: 84559715 | EMD:  | Due: 
✅ Parsed TID: 84320205 | EMD:  | Due: 
✅ Parsed TID: 84320223 | EMD:  | Due: 
✅ Parsed TID: 84404879 | EMD:  | Due: 
✅ Parsed TID: 84346627 | EMD:  | Due: 
✅ Parsed TID: 79322482 | EMD:  | Due: 
✅ Parsed TID: 84215666 | EMD:  | Due: 
✅ Parsed TID: 84521969 | EMD:  | Due: 
✅ Parsed TID: 84320168 | EMD:  | Due: 
✅ Parsed TID: 84339901 | EMD:  | Due: 
✅ Parsed TID: 84343961 | EMD:  | Due: 
✅ Parsed TID: 84563657 | EMD:  | Due: 
✅ Parsed TID: 84671403 | EMD:  | Due: 
✅ Parsed TID: 84339761 | EMD:  | Due: 
✅ Parsed TID: 84346463 | EMD:  | Due: 
✅ Parsed TID: 84648825 | EMD:  | Due: 
✅ Parsed TID: 84575538 | EMD:  | Due: 
✅ Parsed TID: 84207160 | EMD:  | Due: 
✅ Parsed TID: 84471368 | EMD:  |

In [17]:
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from pathlib import Path

# ChromeDriver Setup
driver_path = r"C:\Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.68\chromedriver.exe"
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(driver_path), options=options)

# Load TenderTiger Page
url = "https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=odisha-tenders"
driver.get(url)
time.sleep(5)
html = driver.page_source
driver.quit()

# Parse HTML
soup = BeautifulSoup(html, "html.parser")
tags = soup.find_all("div")

tenders = []
seen_tids = set()

print(f"🔍 Scanning {len(tags)} <div> blocks")

for tag in tags:
    text = tag.get_text(separator="\n").strip()
    if "TID:" not in text:
        continue

    # Clean extra whitespace
    text = re.sub(r"\s+", " ", text)

    # Extract key fields using REGEX
    tid_match = re.search(r"TID[:\s]*(\d+)", text)
    desc_match = re.search(r"Tender Invited For (.*?) Qty", text)
    financials = re.search(r"Worth\s*:([^\nEMD]+)EMD\s*:([^\nDue]+)Due Date\s*:([^\n]+)", text)

    # Sometimes location follows the TID directly
    location_match = re.search(r"TID[:\s]*\d+\s*(.*?)(?:GeM|NewGeM|NCB|COR)", text)

    if tid_match:
        tid = tid_match.group(1).strip()
        if tid in seen_tids:
            continue
        seen_tids.add(tid)

        description = desc_match.group(1).strip() if desc_match else ""
        location = location_match.group(1).strip() if location_match else ""
        worth = financials.group(1).strip() if financials else ""
        emd = financials.group(2).strip() if financials else ""
        due_date = financials.group(3).strip() if financials else ""

        tenders.append({
            "TID": tid,
            "Location": location,
            "Description": description,
            "Worth": worth,
            "EMD": emd,
            "Due Date": due_date
        })

        print(f"✅ TID {tid} | Worth: {worth} | EMD: {emd} | Due: {due_date}")

# Save to Excel
if tenders:
    df = pd.DataFrame(tenders)
    df.to_excel("odisha_tenders_final_parsed.xlsx", index=False)
    print(f"\n✅ Saved {len(tenders)} unique tenders → 'odisha_tenders_final_parsed.xlsx'")
else:
    print("⚠️ No tenders extracted.")


🔍 Scanning 2517 <div> blocks
✅ TID 83934997 | Worth: INR 3.94 Cr | EMD: INR 19.71 Lac | Due: 16 June 2025 3 Days to go me this Tender Invalid mobile number Invalid email address Select Country Please selcet City Whatsapp me this Tender 100.00% Buy for 750 Points 2 TID: 83973694 Ganjam, Orissa (odisha), India GeM PQ COR DOC NCB Tender Invited For laboratory press (q3) Qty:1 Worth : Refer Document EMD : Refer Document Due Date : 19 June 2025 6 Days to go me this Tender Invalid mobile number Invalid email address Select Country Please selcet City Whatsapp me this Tender 98.52% Buy for 500 Points 3 TID: 83927580 Khordha, Orissa (odisha), India New GeM COR DOC NCB Tender Invited For Allen Key set of 12 pieces, Calliper inside with spring, Callipers outside with spring, Center Punch, Dividers with spring, Electrician Screw Driver, Hammer ball peen with handle, Hands file for Second cut flat, Philips Screw Driver set of 5 pieces, Pliers combination, Screw driver Blade, Scriber, Spanner DE set

In [19]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from pathlib import Path

# ✅ SETUP ChromeDriver
driver_path = r"C:\Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.68\chromedriver.exe"
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(driver_path), options=options)

# ✅ OPEN TenderTiger URL
url = "https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=odisha-tenders"
driver.get(url)
time.sleep(5)  # wait for JS content

# ✅ PARSE page
html = driver.page_source
driver.quit()
soup = BeautifulSoup(html, "html.parser")
divs = soup.find_all("div")

tenders = []
seen_tids = set()

# ✅ EXTRACT each <div> block
for div in divs:
    block = div.get_text(separator="\n").strip()
    if "TID:" not in block:
        continue

    lines = [line.strip() for line in block.split("\n") if line.strip()]
    tid = location = desc = worth = emd = due = ""

    for i, line in enumerate(lines):
        if line.startswith("TID:"):
            parts = line.replace("TID:", "").strip().split()
            if not parts:
                continue  # skip malformed lines
            tid = parts[0]
            location = " ".join(parts[1:]).strip()
        elif "Tender Invited For" in line:
            desc = line.split("Tender Invited For")[-1].strip()
        elif "Worth :" in line and "EMD :" in line and "Due Date :" in line:
            match = re.search(r"Worth\s*:(.*?)EMD\s*:(.*?)Due Date\s*:(.*?)(\s|$)", line)
            if match:
                worth = match.group(1).strip()
                emd = match.group(2).strip()
                due = match.group(3).strip()

    if tid and tid not in seen_tids:
        seen_tids.add(tid)
        tenders.append({
            "TID": tid,
            "Location": location,
            "Description": desc,
            "Worth": worth,
            "EMD": emd,
            "Due Date": due
        })
        print(f"✅ TID {tid} | Location: {location} | Worth: {worth} | EMD: {emd} | Due: {due}")

# ✅ SAVE TO EXCEL
if tenders:
    df = pd.DataFrame(tenders)
    output_file = "odisha_tenders_cleaned.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Extracted {len(tenders)} unique tenders. Saved to: {output_file}")
else:
    print("⚠️ No tender entries found.")


⚠️ No tender entries found.


In [20]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time

# Setup Chrome
driver_path = r"C:\Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.68\chromedriver.exe"
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(driver_path), options=options)

# Open URL
driver.get("https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=odisha-tenders")
time.sleep(5)
html = driver.page_source
driver.quit()

# Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")

# 🔍 Try to locate tender list table
table_container = soup.find("div", class_="TenderList-table")
if not table_container:
    print("❌ Could not find .TenderList-table container.")
else:
    rows = table_container.find_all("div", class_="row")
    print(f"\n🔍 Found {len(rows)} rows inside .TenderList-table")

    for idx, row in enumerate(rows):
        block_text = row.get_text(separator="\n").strip()
        print(f"\n--- Tender Block {idx+1} ---")
        print(block_text)



🔍 Found 6 rows inside .TenderList-table

--- Tender Block 1 ---
AI  Search 





                                    Search 










Old  Search


Try AI Search


Advance Search

--- Tender Block 2 ---
AI  Search 





                                    Search 








Old  Search


Try AI Search


Advance Search

--- Tender Block 3 ---
AI  Search 










Old  Search


Try AI Search


Advance Search

--- Tender Block 4 ---


--- Tender Block 5 ---
Tenders By State 
























 India City Tenders 
























Tenders By Authority 
























 Tenders By Keyword

--- Tender Block 6 ---
Free Tender 


Enter Mobile Number









                                    This mobile number is not registered in our system.

                                    Please fill following details to download documents.
                                




















Submit 


Submit


In [22]:
import re
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# --- Setup Selenium
driver_path = r"C:\Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.68\chromedriver.exe"
options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless")  # Optional: run without opening a browser window
driver = webdriver.Chrome(service=Service(driver_path), options=options)

# --- Load the TenderTiger page
url = "https://www.tendertiger.com/TenderAI/TenderAIList?searchtext=odisha-tenders"
driver.get(url)
time.sleep(6)
html = driver.page_source
driver.quit()

# --- Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
divs = soup.find_all("div")
tenders = []
seen_tids = set()

print(f"🔍 Scanning {len(divs)} <div> blocks...\n")

for div in divs:
    text = div.get_text(separator="\n").strip()
    if "TID:" not in text:
        continue

    text = re.sub(r"\s+", " ", text)  # Normalize whitespace
    tid_match = re.search(r"TID[:\s]*(\d+)", text)
    desc_match = re.search(r"Tender Invited For\s*(.+?)(Qty|Worth|EMD|Due Date)", text)
    financial_match = re.search(r"Worth\s*:([^\nEMD]+)EMD\s*:([^\nDue]+)Due Date\s*:([^\n]+)", text)
    qty_match = re.search(r"Qty\s*:\s*([\d,]+)", text)
    match_pct = re.search(r"(\d{1,3}\.\d{2})%", text)

    if tid_match:
        tid = tid_match.group(1)
        if tid in seen_tids:
            continue
        seen_tids.add(tid)

        description = desc_match.group(1).strip() if desc_match else ""
        worth = financial_match.group(1).strip() if financial_match else ""
        emd = financial_match.group(2).strip() if financial_match else ""
        due_date = financial_match.group(3).strip() if financial_match else ""
        qty = qty_match.group(1).strip() if qty_match else ""
        match_percent = match_pct.group(1) + "%" if match_pct else ""

        # Optional: Extract location heuristically (after TID and before 'GeM' or 'India')
        location_match = re.search(r"TID:\d+\s+([^,]+,[^)]+)", text)
        location = location_match.group(1).strip() if location_match else ""

        tenders.append({
            "TID": tid,
            "Location": location,
            "Description": description,
            "Quantity": qty,
            "Worth": worth,
            "EMD": emd,
            "Due Date": due_date,
            "Match %": match_percent
        })

        print(f"✅ {tid} | {description[:50]}... | ₹{worth} | EMD: {emd} | Due: {due_date}")

# --- Save to Excel
if tenders:
    df = pd.DataFrame(tenders)
    df.to_excel("odisha_tenders_final_clean.xlsx", index=False)
    print(f"\n✅ Saved {len(tenders)} tenders → 'odisha_tenders_final_clean.xlsx'")
else:
    print("⚠️ No tender data found.")


🔍 Scanning 2517 <div> blocks...

✅ 83934997 | Manpower Outsourcing Services - Fixed Remuneration... | ₹INR 3.94 Cr | EMD: INR 19.71 Lac | Due: 16 June 2025 3 Days to go me this Tender Invalid mobile number Invalid email address Select Country Please selcet City Whatsapp me this Tender 100.00% Buy for 750 Points 2 TID: 83973694 Ganjam, Orissa (odisha), India GeM PQ COR DOC NCB Tender Invited For laboratory press (q3) Qty:1 Worth : Refer Document EMD : Refer Document Due Date : 19 June 2025 6 Days to go me this Tender Invalid mobile number Invalid email address Select Country Please selcet City Whatsapp me this Tender 98.52% Buy for 500 Points 3 TID: 83927580 Khordha, Orissa (odisha), India New GeM COR DOC NCB Tender Invited For Allen Key set of 12 pieces, Calliper inside with spring, Callipers outside with spring, Center Punch, Dividers with spring, Electrician Screw Driver, Hammer ball peen with handle, Hands file for Second cut flat, Philips Screw Driver set of 5 pieces, Pliers combin